
# Tutorial 7b: Data Imputation

This covers:

* The deletion approach
    - Deleting the incomplete features
    - Deleting the incomplete instances

* pandas
    - Simple imputation using pandas
    - Interpolation imputation using pandas
    
* sklearn
    - Simple imputation using sklearn
    - KNN-based imputation using skearn
    - Iterative imputation using skearn

* Applying the learned models to incomplete test data

----

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Loading and exploring the data

In [2]:
import pandas as pd
# Bộ Titanic đầy đủ (1309 hành khách, có nhãn) - cùng dữ liệu với OpenML.
# Ưu tiên tải từ OpenML; nếu không truy cập được internet thì đọc bản sao cục bộ titanic_openml.csv
# (Kaggle test.csv không có nhãn nên không dùng ở đây, ta tự chia train/test sau).
try:
    titanic_data = pd.read_csv("https://www.openml.org/data/get_csv/16826755/phpMYEkMl.csv", na_values=['?'])
except Exception as e:
    print("Không tải được từ OpenML, dùng file cục bộ:", type(e).__name__)
    titanic_data = pd.read_csv("titanic_openml.csv").dropna(how="all")
    titanic_data["survived"] = titanic_data["survived"].astype(int)
    titanic_data["pclass"] = titanic_data["pclass"].astype(int)
    titanic_data["sibsp"] = titanic_data["sibsp"].astype(int)
    titanic_data["parch"] = titanic_data["parch"].astype(int)
titanic_data.head()

Không tải được từ OpenML, dùng file cục bộ: HTTPError


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


**Values considered “missing”**

There are many ways to represent missing values in both the dataset file and the python pandas.

Missing values in the data might be blank entries, or '?', or something else that data collecters agreed on to represent unobserved data.
In this case it is '?' -- knowing this, we tell `pandas` what to consider as missing values via `na_values=['?']`.

At the "other end", `pandas` can represent missing values in several different ways. As can be seen above, "NaN" is the default missing value marker, however, we need to be able to easily detect this value with data of different types: floating point, integer, boolean, and general object. In many cases, however, some other forms can refer to missing values such as None “missing” or “not available”, “NA", or (-)inf .


In [3]:
# Let's drop some features that we will not consider here.
titanic_data.drop(['name','ticket', 'embarked', 'boat' ,'body' ,'home.dest'], axis=1, inplace=True)

Now we will split the data to train and test subsets as **ONLY** the training data will be used to learn the imputers then the learnt models are applied to the test data

In [4]:
from sklearn.model_selection import train_test_split
y=titanic_data['survived']
X=titanic_data.drop(['survived'], axis=1)
X_titanic_train, X_titanic_test, y_titanic_train, y_titanic_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [5]:
X_titanic_train

,pclass,sex,age,sibsp,parch,fare,cabin
1214,3,male,NaN,0,0,8.6625,NaN
677,3,male,26.0,0,0,7.8958,NaN
534,2,female,19.0,0,0,26.0000,NaN
1174,3,female,NaN,8,2,69.5500,NaN
864,3,female,28.0,0,0,7.7750,NaN
...,...,...,...,...,...,...,...
1095,3,female,NaN,0,0,7.6292,NaN
1130,3,female,18.0,0,0,7.7750,NaN
1294,3,male,28.5,0,0,16.1000,NaN
860,3,female,26.0,0,0,7.9250,NaN


In [6]:
#Now if we perform classification it might not work for most classifiers
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()
#classifier=SVC()
classifier.fit(X_titanic_train, y_titanic_train)


ValueError: could not convert string to float: 'male'

In [7]:
X_titanic_train.isna().sum(), len(X_titanic_train)

(pclass      0
 sex         0
 age       187
 sibsp       0
 parch       0
 fare        1
 cabin     712
 dtype: int64,
 916)

# There is a problem that some features contain string values, namely the features "sex" and "cabin", so lets encode these features

In [8]:
# Cần scikit-learn đủ mới để OrdinalEncoder hỗ trợ handle_unknown='use_encoded_value' với unknown_value=np.nan.
# (Đã bỏ lệnh "!pip install -U scikit-learn" vì nó nâng cấp thư viện ngay giữa phiên chạy và làm hỏng các import phía sau;
#  nếu môi trường của bạn cũ, hãy chạy lệnh đó MỘT LẦN rồi khởi động lại kernel.)
import sklearn
print('The scikit-learn version is {}.'.format(sklearn.__version__))

The scikit-learn version is 1.8.0.


In [9]:
import numpy as np
# Encoding categorical features with preserving the missing values in incomplete features
from sklearn.preprocessing import OrdinalEncoder
encoder_sex = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value=np.nan)
X_titanic_train_encoded=X_titanic_train.copy()
X_titanic_train_encoded['sex'] = encoder_sex.fit_transform(X_titanic_train_encoded['sex'].values.reshape(-1, 1))

#Now lets encode the incomplete Cabin feature
encoder_cabin = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value=np.nan) #You can use the same encoder for both but we use two for the sake of clarfication
X_titanic_train_encoded['cabin'] = encoder_cabin.fit_transform(X_titanic_train_encoded['cabin'].values.reshape(-1, 1).astype(str))
#get the code of the "nan" value for the cabin categorical feature
cabin_nan_code=encoder_cabin.transform([['nan']])[0][0]
#print(cabin_nan_code)
#Now, retrive the nan values to be missing in the encoded data
X_titanic_train_encoded['cabin'] = X_titanic_train_encoded['cabin'].replace(cabin_nan_code, np.nan)   # gán lại tường minh (an toàn với pandas>=2/3)

print('cabin còn thiếu sau khi mã hóa:', X_titanic_train_encoded['cabin'].isna().sum(), '/', len(X_titanic_train_encoded))

cabin còn thiếu sau khi mã hóa: 712 / 916


In [10]:
np.unique(X_titanic_train_encoded['cabin'])

array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
       110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
       121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
       132., 133., 134., 135., 136., 137., 138., 139., 140., 141., 142.,
       143., 144., 145.,  nan])

In [11]:
np.unique(X_titanic_train_encoded['sex'])

array([0., 1.])

In [12]:
np.unique(X_titanic_train_encoded['age'])

array([ 0.1667,  0.3333,  0.75  ,  0.8333,  0.9167,  1.    ,  2.    ,
        3.    ,  4.    ,  5.    ,  6.    ,  7.    ,  8.    ,  9.    ,
       10.    , 11.    , 11.5   , 12.    , 13.    , 14.    , 14.5   ,
       15.    , 16.    , 17.    , 18.    , 18.5   , 19.    , 20.    ,
       21.    , 22.    , 22.5   , 23.    , 23.5   , 24.    , 24.5   ,
       25.    , 26.    , 26.5   , 27.    , 28.    , 28.5   , 29.    ,
       30.    , 30.5   , 31.    , 32.    , 32.5   , 33.    , 34.    ,
       34.5   , 35.    , 36.    , 36.5   , 37.    , 38.    , 38.5   ,
       39.    , 40.    , 40.5   , 41.    , 42.    , 43.    , 44.    ,
       45.    , 46.    , 47.    , 48.    , 49.    , 50.    , 51.    ,
       52.    , 53.    , 54.    , 55.    , 55.5   , 56.    , 57.    ,
       58.    , 59.    , 60.    , 60.5   , 61.    , 62.    , 63.    ,
       64.    , 65.    , 71.    , 76.    , 80.    ,     nan])

## `X_titanic_train_encoded` is the encoded incomplete training data

In [13]:
#Check the types of the encoded data, no object features
X_titanic_train_encoded.info()

<class 'pandas.DataFrame'>
Index: 916 entries, 1214 to 1126
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  916 non-null    int64  
 1   sex     916 non-null    float64
 2   age     729 non-null    float64
 3   sibsp   916 non-null    int64  
 4   parch   916 non-null    int64  
 5   fare    915 non-null    float64
 6   cabin   204 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 57.2 KB


In [14]:
# As the data has no strings/object snow, let's try performing classification using the encoded data
classifier.fit(X_titanic_train_encoded, y_titanic_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

## Note the error:ValueError: Input contains NaN, infinity or a value too large for dtype('float32').

We need to handle the missing values before performing the classification.

Lets show the number of missing values in each feature of the encoded train data



In [15]:
print("The number of missing values ")
print(X_titanic_train_encoded.isnull().sum())

The number of missing values 
pclass      0
sex         0
age       187
sibsp       0
parch       0
fare        1
cabin     712
dtype: int64


We have three incomplete features "age", "fare", and "cabin"

## The deletion approach

### Deleting the incomplete features

In [16]:
X_titanic_train_complete=X_titanic_train_encoded.copy()
X_titanic_train_complete.dropna(axis=1, inplace=True)
X_titanic_train_complete

,pclass,sex,sibsp,parch
1214,3,1.0,0,0
677,3,1.0,0,0
534,2,0.0,0,0
1174,3,0.0,8,2
864,3,0.0,0,0
...,...,...,...,...
1095,3,0.0,0,0
1130,3,0.0,0,0
1294,3,1.0,0,0
860,3,0.0,0,0


In [17]:
#Check the number of missing values
print(X_titanic_train_complete.isnull().sum())

pclass    0
sex       0
sibsp     0
parch     0
dtype: int64


### Deleting the incomplete instances

In [18]:
X_titanic_train_complete=X_titanic_train_encoded.copy()
X_titanic_train_complete.dropna(axis=0, inplace=True)
#The difference is axis=0 instead of 1
X_titanic_train_complete

,pclass,sex,age,sibsp,parch,fare,cabin
39,1,1.0,48.0,0,0,50.4958,14.0
30,1,1.0,45.0,0,0,35.5000,145.0
242,1,0.0,33.0,0,0,27.7208,0.0
136,1,1.0,53.0,0,0,28.5000,68.0
3,1,1.0,30.0,1,2,151.5500,61.0
...,...,...,...,...,...,...,...
189,1,1.0,29.0,0,0,30.0000,113.0
252,1,1.0,61.0,1,3,262.3750,35.0
21,1,0.0,47.0,1,1,52.5542,101.0
276,1,1.0,57.0,1,0,146.5208,42.0


## Notice the reduction in the number of instances

Another important point for the instance deletion approach is that there is a need to remove the target values (from y_train) that correspond to the incomplete (deleted) data instances

In [19]:
#Check the number of missing values
print(X_titanic_train_complete.isnull().sum())

pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


The deletion approach has several drawbacks. It reduces the availlable data, which limits the learning ability, especially when there are many missing values.

Furthermore, the approach of deleting incomplete instances is not practical for test data: we really want to know the answer!

## Imputation using `pandas`

### Simple imputation (`pandas`)

In [20]:
#Mean for numeric values
X_titanic_data_complete=X_titanic_train_encoded.copy()
X_titanic_data_complete['age']=X_titanic_data_complete['age'].fillna(X_titanic_data_complete['age'].mean())
X_titanic_data_complete['fare']=X_titanic_data_complete['fare'].fillna(X_titanic_data_complete['fare'].mean())
X_titanic_data_complete['cabin']=X_titanic_data_complete['cabin'].fillna(X_titanic_data_complete['cabin'].mean())
# Show the number of missing values
print(X_titanic_data_complete.isnull().sum())

pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


In [21]:
X_titanic_data_complete.head()

,pclass,sex,age,sibsp,parch,fare,cabin
1214,3,1.0,29.102309,0,0,8.6625,73.27451
677,3,1.0,26.000000,0,0,7.8958,73.27451
534,2,0.0,19.000000,0,0,26.0000,73.27451
1174,3,0.0,29.102309,8,2,69.5500,73.27451
864,3,0.0,28.000000,0,0,7.7750,73.27451


## "interpolation" (`pandas`)

In [22]:
X_titanic_data_complete = X_titanic_train_encoded.copy()
X_titanic_data_complete = X_titanic_data_complete.interpolate()
#The output is 'numpy.ndarray' so we convert it to dataframe for consistency
X_titanic_train_complete = pd.DataFrame(X_titanic_train_complete)
print(X_titanic_train_complete.isna().sum())

pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


## Imputation using `sklearn`

### Simple imputation (`sklearn`)

In [23]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer()
X_titanic_train_complete = imputer.fit_transform(X_titanic_train_encoded)
#The output is 'numpy.ndarray' so we convert it to dataframe for consistency
X_titanic_train_complete=pd.DataFrame(X_titanic_train_complete, columns=X_titanic_train_encoded.columns)
print("The number of missing values :\n", X_titanic_train_complete.isnull().sum())

The number of missing values :
 pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


In [24]:
X_titanic_train_encoded

,pclass,sex,age,sibsp,parch,fare,cabin
1214,3,1.0,NaN,0,0,8.6625,NaN
677,3,1.0,26.0,0,0,7.8958,NaN
534,2,0.0,19.0,0,0,26.0000,NaN
1174,3,0.0,NaN,8,2,69.5500,NaN
864,3,0.0,28.0,0,0,7.7750,NaN
...,...,...,...,...,...,...,...
1095,3,0.0,NaN,0,0,7.6292,NaN
1130,3,0.0,18.0,0,0,7.7750,NaN
1294,3,1.0,28.5,0,0,16.1000,NaN
860,3,0.0,26.0,0,0,7.9250,NaN


## The default strategy for sklearn simple imputer is the "mean", you can change it using the strategy parameter

In [25]:
imputer = SimpleImputer(strategy="median")
X_titanic_train_complete = imputer.fit_transform(X_titanic_train_encoded)
#The output is 'numpy.ndarray' so we convert it to dataframe for consistency
X_titanic_train_complete=pd.DataFrame(X_titanic_train_complete, columns=X_titanic_train_encoded.columns)
print("The number of missing values :\n", X_titanic_train_complete.isnull().sum())

The number of missing values :
 pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


## kNN imputer (`sklearn`)

In [26]:
from sklearn.impute import KNNImputer
imputer = KNNImputer()
X_titanic_train_complete = imputer.fit_transform(X_titanic_train_encoded)
#The output is 'numpy.ndarray' so we convert it to dataframe for consistency
X_titanic_train_complete=pd.DataFrame(X_titanic_train_complete, columns=X_titanic_train_encoded.columns)
print("The number of missing values :\n", X_titanic_train_complete.isnull().sum())

The number of missing values :
 pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


In [27]:
#The default k for the KNN imputer is 5, you can change it as follows:
imputer = KNNImputer(n_neighbors=2)
# etc etc...

## Iterative Imputer (`sklearn`)

Note this is sklearn's implementation of a method originally known as "MICE" -- see lecture 2 from this week for an explanation.

In [28]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
imputer = IterativeImputer()
X_titanic_train_complete = imputer.fit_transform(X_titanic_train_encoded)
#The output is 'numpy.ndarray' so we convert it to dataframe for consistency
X_titanic_train_complete=pd.DataFrame(X_titanic_train_complete, columns=X_titanic_train_encoded.columns)
print("The number of missing values :\n", X_titanic_train_complete.isnull().sum())

The number of missing values :
 pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


You can reset the default parameters of the iterative imputer. For example, you can set the number of iterations. Moreover, you can specify the estimator for estimating the missing values.

In [29]:
# Lets use DT as an estimator
from sklearn.tree import DecisionTreeRegressor
imputer = IterativeImputer(estimator=DecisionTreeRegressor())
X_titanic_train_complete = imputer.fit_transform(X_titanic_train_encoded)
#The output is 'numpy.ndarray' so we convert it to dataframe for consistency
X_titanic_train_complete=pd.DataFrame(X_titanic_train_complete, columns=X_titanic_train_encoded.columns)
print("The number of missing values :\n", X_titanic_train_complete.isnull().sum())

The number of missing values :
 pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


## Applying the learned models to incomplete test data

First, apply the encoders

In [30]:
#The learnt encoder_sex should be used to encode the test data, NOTE there is NO fit here, just transform
X_titanic_test_encoded=X_titanic_test.copy()
X_titanic_test_encoded['sex'] = encoder_sex.transform(X_titanic_test_encoded['sex'].values.reshape(-1, 1))

#The learnt encoder2 should be used to encode the test data, NOTE there is NO fit here, just transform
X_titanic_test_encoded['cabin'] = encoder_cabin.transform(X_titanic_test_encoded['cabin'].values.reshape(-1, 1).astype(str))
#Now, retrive the nan values to be missing in the encoded data
X_titanic_test_encoded['cabin'] = X_titanic_test_encoded['cabin'].replace(cabin_nan_code, np.nan)   # gán lại tường minh (an toàn với pandas>=2/3)


Second, use the learned imputer to estimate the missing values in the test data

In [31]:
print("The number of missing values in the test data before imputation :\n", X_titanic_test_encoded.isnull().sum())
X_titanic_test_complete = imputer.transform(X_titanic_test_encoded)
X_titanic_test_complete=pd.DataFrame(X_titanic_test_complete, columns=X_titanic_test_encoded.columns)
print("The number of missing values in the test data after imputation :\n", X_titanic_test_complete.isnull().sum())

The number of missing values in the test data before imputation :
 pclass      0
sex         0
age        76
sibsp       0
parch       0
fare        0
cabin     349
dtype: int64
The number of missing values in the test data after imputation :
 pclass    0
sex       0
age       0
sibsp     0
parch     0
fare      0
cabin     0
dtype: int64


Finally, we can perform the classification using the imputed complete data.

In [32]:
#We use f-measure because the classes are not balanced
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(random_state=0)
#classifier=SVC()
classifier.fit(X_titanic_train_complete, y_titanic_train)
print("F1 score after imputation = ", f1_score(classifier.predict(X_titanic_test_complete), y_titanic_test))

F1 score after imputation =  0.7278481012658228


----